# Notebook: Prepare, Clean, Merge, and Split the Datasets

**Purpose:** This notebook explains the work performed by `src/prepare_data.py` in small, inspectable steps.

**Tasks kept separate:**
- Jigsaw has six toxicity labels.
- SMS has `ham` / `spam` labels.
- Missing labels for the other task stay empty (`NaN`), not zero.

## 1. Find the project folder and import libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re
import json
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data folder exists:", RAW.exists())
print("Processed data folder:", OUT)

Project root: /Users/kakulrajput/Documents/Smart_Community_ML
Raw data folder exists: True
Processed data folder: /Users/kakulrajput/Documents/Smart_Community_ML/data/processed


## 2. Define conservative text cleaning
normalize Unicode, remove invisible zero-width/BOM characters, collapse repeated whitespace, and trim the ends. We preserve punctuation, case, and negation because they may carry meaning for moderation.

In [3]:
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

def clean_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", value).strip()

examples = pd.DataFrame({
    "before": ["  Hello   world! ", "This\u200b is fine", "don't remove punctuation!!!"],
})
examples["after"] = examples["before"].map(clean_text)
display(examples)

,before,after
0,Hello world!,Hello world!
1,This​ is fine,This is fine
2,don't remove punctuation!!!,don't remove punctuation!!!


## 3. Load and clean Jigsaw
The Jigsaw training CSV should contain `id`, `comment_text`, and six toxicity columns. We rename `comment_text` to the shared `text` column.

In [4]:
jigsaw_path = RAW / "jigsaw" / "train.csv"
jigsaw_raw = pd.read_csv(jigsaw_path)

print("Raw Jigsaw shape:", jigsaw_raw.shape)
print("Columns:", jigsaw_raw.columns.tolist())
display(jigsaw_raw.head(3))

Raw Jigsaw shape: (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0


In [5]:
required = ["id", "comment_text"] + LABELS
missing = [column for column in required if column not in jigsaw_raw.columns]
if missing:
    raise ValueError(f"Jigsaw dataset is missing required columns: {missing}")

jigsaw = jigsaw_raw[required].copy()
jigsaw["text"] = jigsaw["comment_text"].map(clean_text)
jigsaw = jigsaw.drop(columns=["comment_text"])
before_rows = len(jigsaw)
jigsaw = jigsaw[jigsaw["text"].ne("")]
jigsaw = jigsaw.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

for label in LABELS:
    jigsaw[label] = pd.to_numeric(jigsaw[label], errors="coerce").fillna(0).astype("int8")

jigsaw["source"] = "jigsaw"
jigsaw["spam_label"] = np.nan
jigsaw["spam"] = np.nan
jigsaw = jigsaw[["id", "text", "source"] + LABELS + ["spam_label", "spam"]]

print("Rows before removing blank/duplicate text:", before_rows)
print("Rows after cleaning:", len(jigsaw))
display(jigsaw.head(5))

Rows before removing blank/duplicate text: 159571
Rows after cleaning: 159374


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,0000997932d777bf,Explanation Why the edits made under my userna...,jigsaw,0,0,0,0,0,0,NaN,NaN
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,jigsaw,0,0,0,0,0,0,NaN,NaN
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",jigsaw,0,0,0,0,0,0,NaN,NaN
3,0001b41b1c6bb37e,""" More I can't make any real suggestions on im...",jigsaw,0,0,0,0,0,0,NaN,NaN
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",jigsaw,0,0,0,0,0,0,NaN,NaN


## 4. Load and clean SMS Spam Collection
The SMS file is tab-separated and typically has no header. Each valid row contains a label (`ham` or `spam`) followed by the message.

In [6]:
sms_path = RAW / "sms" / "SMSSpamCollection"
sms_rows = []

with open(sms_path, encoding="utf-8", errors="replace") as file:
    for line in file:
        line = line.rstrip("\r\n")
        if "\t" not in line:
            continue
        label, message = line.split("\t", 1)
        label = label.strip().lower()
        if label in {"ham", "spam"}:
            sms_rows.append((message, label))

sms = pd.DataFrame(sms_rows, columns=["text", "spam_label"])
print("Parsed SMS rows:", len(sms))
display(sms.head(5))

Parsed SMS rows: 5574


,text,spam_label
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [7]:
sms["text"] = sms["text"].map(clean_text)
sms = sms[sms["text"].ne("")]
sms = sms.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
sms["id"] = [f"sms_{i:06d}" for i in range(len(sms))]
sms["source"] = "sms"
sms["spam"] = (sms["spam_label"] == "spam").astype("int8")

for label in LABELS:
    sms[label] = np.nan

sms = sms[["id", "text", "source"] + LABELS + ["spam_label", "spam"]]
print("SMS rows after cleaning:", len(sms))
display(sms.head(5))
display(sms["spam_label"].value_counts().rename("rows").to_frame())

SMS rows after cleaning: 5160


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,sms_000000,"Go until jurong point, crazy.. Available only ...",sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
1,sms_000001,Ok lar... Joking wif u oni...,sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
2,sms_000002,Free entry in 2 a wkly comp to win FA Cup fina...,sms,NaN,NaN,NaN,NaN,NaN,NaN,spam,1
3,sms_000003,U dun say so early hor... U c already then say...,sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
4,sms_000004,"Nah I don't think he goes to usf, he lives aro...",sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0


,rows
spam_label,
ham,4518
spam,642


## 5. Merge the cleaned datasets
The shared schema makes the data easier to inspect together. `source` identifies where each row came from. We do **not** equate spam with toxicity.

In [8]:
master = pd.concat([jigsaw, sms], ignore_index=True)

jigsaw.to_csv(OUT / "jigsaw_cleaned.csv", index=False)
sms.to_csv(OUT / "sms_cleaned.csv", index=False)
master.to_csv(OUT / "unified_master.csv", index=False)

print("Jigsaw cleaned:", len(jigsaw))
print("SMS cleaned:", len(sms))
print("Unified master:", len(master))
display(master.groupby("source").size().rename("rows").to_frame())
display(master.head(5))

Jigsaw cleaned: 159374
SMS cleaned: 5160
Unified master: 164534


,rows
source,
jigsaw,159374
sms,5160


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,0000997932d777bf,Explanation Why the edits made under my userna...,jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,0001b41b1c6bb37e,""" More I can't make any real suggestions on im...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


## 6. Create independent train / validation / test splits
Each task is split separately. We use 80% training, 10% validation, and 10% testing. Jigsaw is stratified on its main `toxic` label because rare combinations of all six labels cannot all be reliably stratified. SMS is stratified on `ham` / `spam`.

In [9]:
tox_train, tox_temp = train_test_split(
    jigsaw, test_size=0.20, random_state=42, stratify=jigsaw["toxic"]
)
tox_validation, tox_test = train_test_split(
    tox_temp, test_size=0.50, random_state=42, stratify=tox_temp["toxic"]
)

spam_train, spam_temp = train_test_split(
    sms, test_size=0.20, random_state=42, stratify=sms["spam_label"]
)
spam_validation, spam_test = train_test_split(
    spam_temp, test_size=0.50, random_state=42, stratify=spam_temp["spam_label"]
)

splits = {
    "toxicity_train.csv": tox_train,
    "toxicity_validation.csv": tox_validation,
    "toxicity_test.csv": tox_test,
    "spam_train.csv": spam_train,
    "spam_validation.csv": spam_validation,
    "spam_test.csv": spam_test,
}
for filename, frame in splits.items():
    frame.to_csv(OUT / filename, index=False)

split_summary = pd.DataFrame([
    {"task": "toxicity", "split": name, "rows": len(frame)}
    for name, frame in [
        ("train", tox_train), ("validation", tox_validation), ("test", tox_test)
    ]
] + [
    {"task": "spam", "split": name, "rows": len(frame)}
    for name, frame in [
        ("train", spam_train), ("validation", spam_validation), ("test", spam_test)
    ]
])
display(split_summary)

,task,split,rows
0,toxicity,train,127499
1,toxicity,validation,15937
2,toxicity,test,15938
3,spam,train,4128
4,spam,validation,516
5,spam,test,516


## 7. Write a data manifest
The manifest records the important preparation choices and row counts so the workflow is reproducible.

In [10]:
manifest = {
    "jigsaw_cleaned_rows": int(len(jigsaw)),
    "sms_cleaned_rows": int(len(sms)),
    "unified_master_rows": int(len(master)),
    "normalization": "NFKC; remove zero-width/BOM; collapse whitespace; trim; preserve punctuation, case, and negation",
    "label_policy": "Task labels remain separate; missing labels are null, not zero.",
    "split_policy": "Independent 80/10/10 split, random_state=42."
}
(OUT / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

{
  "jigsaw_cleaned_rows": 159374,
  "sms_cleaned_rows": 5160,
  "unified_master_rows": 164534,
  "normalization": "NFKC; remove zero-width/BOM; collapse whitespace; trim; preserve punctuation, case, and negation",
  "label_policy": "Task labels remain separate; missing labels are null, not zero.",
  "split_policy": "Independent 80/10/10 split, random_state=42."
}


## 8. Final verification
- `jigsaw_cleaned.csv`, `sms_cleaned.csv`, and `unified_master.csv` exist.
- Six toxicity and spam split CSVs exist.
- `data_manifest.json` was created.
- Raw data files remain unchanged.

In [11]:
expected_files = [
    "jigsaw_cleaned.csv", "sms_cleaned.csv", "unified_master.csv",
    "toxicity_train.csv", "toxicity_validation.csv", "toxicity_test.csv",
    "spam_train.csv", "spam_validation.csv", "spam_test.csv", "data_manifest.json"
]
verification = pd.DataFrame([
    {"file": name, "exists": (OUT / name).exists(),
     "size_kb": round((OUT / name).stat().st_size / 1024, 1) if (OUT / name).exists() else None}
    for name in expected_files
])
display(verification)
assert verification["exists"].all(), "One or more expected output files are missing."
print("Preparation notebook completed successfully.")

,file,exists,size_kb
0,jigsaw_cleaned.csv,True,67607.6
1,sms_cleaned.csv,True,543.6
2,unified_master.csv,True,70028.9
3,toxicity_train.csv,True,54228.8
4,toxicity_validation.csv,True,6631.7
5,toxicity_test.csv,True,6747.3
6,spam_train.csv,True,435.0
7,spam_validation.csv,True,53.0
8,spam_test.csv,True,55.7
9,data_manifest.json,True,0.4


Preparation notebook completed successfully.
